### RAG Pipeline - Dat INgestion to Vector DB

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  
from pathlib import Path

C:\Users\amrit\AppData\Local\Temp\ipykernel_37780\4142968636.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Amrit-Mahajan-FlowCV-Cover-Letter.pdf
  ✓ Loaded 1 pages

Processing: Resume_Amrit_Mahajan_GenAI.pdf
  ✓ Loaded 1 pages

Processing: SSR_TSRPT.pdf
  ✓ Loaded 1 pages

Total documents loaded: 3


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m147', 'creator': 'FlowCV - https://flowcv.com', 'creationdate': '2026-05-27T18:42:07+00:00', 'moddate': '2026-05-27T18:42:07+00:00', 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'source': '..\\data\\pdf\\Amrit-Mahajan-FlowCV-Cover-Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Amrit-Mahajan-FlowCV-Cover-Letter.pdf', 'file_type': 'pdf'}, page_content="Amrit Mahajan\namrit.mahajan007 @gmail.com\n \n(480)527-3622\n \nTempe, AZ\n \namrit-mahajan\n \nDear Hiring Committee,\nI am expressing my strong interest in the Forward Deployed Engineer role at Cornerstone OnDemand.\nI've completed my Master's in Computer Science at Arizona State University and bring a strong\nfoundation in data science & machine learning, along with a deep passion for technology.\nIn my recent internship at Wells Fargo as a Quantitative Analyst on the GenAI team, I contributed to\nsolutions that streamlined customer data acce

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""] # seperators enable better splitting based on content structure
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 3 documents into 13 chunks

Example chunk:
Content: Amrit Mahajan
amrit.mahajan007 @gmail.com
 
(480)527-3622
 
Tempe, AZ
 
amrit-mahajan
 
Dear Hiring Committee,
I am expressing my strong interest in the Forward Deployed Engineer role at Cornerstone O...
Metadata: {'producer': 'Skia/PDF m147', 'creator': 'FlowCV - https://flowcv.com', 'creationdate': '2026-05-27T18:42:07+00:00', 'moddate': '2026-05-27T18:42:07+00:00', 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'source': '..\\data\\pdf\\Amrit-Mahajan-FlowCV-Cover-Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Amrit-Mahajan-FlowCV-Cover-Letter.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m147', 'creator': 'FlowCV - https://flowcv.com', 'creationdate': '2026-05-27T18:42:07+00:00', 'moddate': '2026-05-27T18:42:07+00:00', 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'source': '..\\data\\pdf\\Amrit-Mahajan-FlowCV-Cover-Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Amrit-Mahajan-FlowCV-Cover-Letter.pdf', 'file_type': 'pdf'}, page_content="Amrit Mahajan\namrit.mahajan007 @gmail.com\n \n(480)527-3622\n \nTempe, AZ\n \namrit-mahajan\n \nDear Hiring Committee,\nI am expressing my strong interest in the Forward Deployed Engineer role at Cornerstone OnDemand.\nI've completed my Master's in Computer Science at Arizona State University and bring a strong\nfoundation in data science & machine learning, along with a deep passion for technology.\nIn my recent internship at Wells Fargo as a Quantitative Analyst on the GenAI team, I contributed to\nsolutions that streamlined customer data acce

### Embedding and vectorestore db

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8637.67it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\amrit\AppData\Local\Temp\ipykernel_37780\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


## Vectorstore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents_cosine", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection with cosine distance
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents_cosine
Existing documents in collection: 0


In [9]:
chunks

[Document(metadata={'producer': 'Skia/PDF m147', 'creator': 'FlowCV - https://flowcv.com', 'creationdate': '2026-05-27T18:42:07+00:00', 'moddate': '2026-05-27T18:42:07+00:00', 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'source': '..\\data\\pdf\\Amrit-Mahajan-FlowCV-Cover-Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Amrit-Mahajan-FlowCV-Cover-Letter.pdf', 'file_type': 'pdf'}, page_content="Amrit Mahajan\namrit.mahajan007 @gmail.com\n \n(480)527-3622\n \nTempe, AZ\n \namrit-mahajan\n \nDear Hiring Committee,\nI am expressing my strong interest in the Forward Deployed Engineer role at Cornerstone OnDemand.\nI've completed my Master's in Computer Science at Arizona State University and bring a strong\nfoundation in data science & machine learning, along with a deep passion for technology.\nIn my recent internship at Wells Fargo as a Quantitative Analyst on the GenAI team, I contributed to\nsolutions that streamlined customer data acce

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 13 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.05it/s]

Generated embeddings with shape: (13, 384)
Adding 13 documents to vector store...
Successfully added 13 documents to vector store
Total documents in collection: 13


In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Rank stored documents directly by cosine similarity.
        try:
            results = self.vector_store.collection.get(
                include=["documents", "metadatas", "embeddings"]
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents']:
                documents = results['documents']
                metadatas = results['metadatas']
                embeddings = np.array(results['embeddings'])
                ids = results['ids']
                similarities = cosine_similarity([query_embedding], embeddings)[0]
                top_indices = np.argsort(similarities)[::-1][:top_k]
                
                for rank, idx in enumerate(top_indices, start=1):
                    retrieved_docs.append({
                        'id': ids[idx],
                        'content': documents[idx],
                        'metadata': metadatas[idx],
                        'cosine_similarity': float(similarities[idx]),
                        'rank': rank
                    })
                
                print(f"Retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("Whats Amrit experience")

Retrieving documents for query: 'Whats Amrit experience'
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 100.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_223c7ebc_3',
  'content': 'Amrit  Mahajan  \n \n480-527-3622  |  amrit.mahajan007@gmail.com |  linkedin.com/in/amrit-mahajan |  lightning-nemesis.github.io  \nEDUCATION  Arizona  State  University  Tempe,  Arizona  Master  of  Science  in  Computer  Science  |  GPA:  4.0/4.0  Aug.  2024  -  May  2026  ●  Relevant  Coursework  -  Natural  Language  Processing,  Semantic  Web  Mining,  Statistical  Machine  Learning,  Data  \nMining,\n \nCloud\n \nComputing,\n \nData-intensive\n \nsystems\n \nfor\n \nML,\n \nData\n \nVisualization\n  \nVellore  Institute  of  Technology,  Vellore  Vellore,  India  Bachelor’s  in  Computer  Science  |  GPA:  9.29/10  |  Merit  Scholarship  Recipient  (Top  10  rank)  Jul.  2018  -  May  2022   \nWORK  EXPERIENCE  Wells  Fargo  Charlotte,  NC  Gen  AI  Intern,  Quantitative  Analytics  Program  June  2025  -  Aug.  2025  ●  Built  a  POC  Generative  AI  Question  Answering  system  using  customer  data,  providing  bankers  with  accurate  \

## RAG Pipeline- VectorDB To LLM Output Generation

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [20]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [21]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [22]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [24]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Is amrit a trainer?")

Retrieving documents for query: 'Is amrit a trainer?'
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_223c7ebc_3',
  'content': 'Amrit  Mahajan  \n \n480-527-3622  |  amrit.mahajan007@gmail.com |  linkedin.com/in/amrit-mahajan |  lightning-nemesis.github.io  \nEDUCATION  Arizona  State  University  Tempe,  Arizona  Master  of  Science  in  Computer  Science  |  GPA:  4.0/4.0  Aug.  2024  -  May  2026  ●  Relevant  Coursework  -  Natural  Language  Processing,  Semantic  Web  Mining,  Statistical  Machine  Learning,  Data  \nMining,\n \nCloud\n \nComputing,\n \nData-intensive\n \nsystems\n \nfor\n \nML,\n \nData\n \nVisualization\n  \nVellore  Institute  of  Technology,  Vellore  Vellore,  India  Bachelor’s  in  Computer  Science  |  GPA:  9.29/10  |  Merit  Scholarship  Recipient  (Top  10  rank)  Jul.  2018  -  May  2022   \nWORK  EXPERIENCE  Wells  Fargo  Charlotte,  NC  Gen  AI  Intern,  Quantitative  Analytics  Program  June  2025  -  Aug.  2025  ●  Built  a  POC  Generative  AI  Question  Answering  system  using  customer  data,  providing  bankers  with  accurate  \

In [ ]:
# 

## Integration Vectordb Context pipeline With LLM output

In [31]:
    ### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3-32b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=5):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [32]:
answer=rag_simple("List companies amrit has worked in",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'List companies amrit has worked in'
Top K: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.12it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents


<think>
Okay, let's see. The user wants me to list the companies Amrit has worked at based on the provided context.

First, I'll look through the context for any mentions of companies. In the WORK EXPERIENCE section, there's Wells Fargo and Bajaj Finance Limited. 

Under WORK EXPERIENCE:
- Wells Fargo in Charlotte, NC as a Gen AI Intern from June 2025 to August 2025.
- Bajaj Finance Limited in Pune, India as a Data Science Intern from January 2022 to July 2022.

Additionally, in the education section, there's Arizona State University and Vellore Institute of Technology, but those are educational institutions, not companies. 

The letter also mentions a Quantitative Analyst role at Wells Fargo again, confirming that. 

So the companies are Wells Fargo and Bajaj Finance Limited. I need to make sure there are no others. The awards section mentions Voxel51 and JP Morgan Chase & Co., but those are related to hackathons, not employment. 

Therefore, the answer should list Wells Fargo and Baj

# Enhanced RAG

In [36]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['cosine_similarity'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['cosine_similarity'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Amrit GPA at VIT Vellore", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Amrit GPA at VIT Vellore'
Top K: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents


Answer: <think>
Okay, let's see. The user is asking for Amrit Mahajan's GPA at VIT Vellore. First, I need to look through the provided context to find the relevant information.

Looking at the transcript, there's a section under "External Degrees" that mentions VIT University with a Bachelor of Technology degree dated 07/01/2022. The GPA isn't directly listed there. However, in the EDUCATION section of the resume part, it says "Vellore Institute of Technology, Vellore Bachelor’s in Computer Science | GPA: 9.29/10 | Merit Scholarship Recipient (Top 10 rank) Jul. 2018 - May 2022". 

Wait, the question is about the GPA at VIT Vellore, which is the same as VIT University. The GPA here is 9.29 out of 10. But sometimes, GPAs can be converted to a 4.0 scale. However, the user didn't ask for a conversion, just the GPA. Since the context provides 9.29/10, that's the answer. The transcript's "External Degrees" section doesn't mention the GPA, but the resume part does. So the answer should be 9.2

In [39]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['cosine_similarity'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what time did amrit work at bajaj finance?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what time did amrit work at bajaj finance?'
Top K: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents
Streaming answer:
Use the following context to answer the question concisely.
Context:
Amrit  Mahajan  
 
480-527-3622  |  amrit.mahajan007@gmail.com |  linkedin.com/in/amrit-mahajan |  lightning-nemesis.github.io  
EDUCATION  Arizona  State  University  Tempe,  Arizona  Master  of  Science  in  Computer  Science  |  GPA:  4.0/4.0  Aug.

  2024  -  May  2026  ●  Relevant  Coursework  -  Natural  Language  Processing,  Semantic  Web  Mining,  Statistical  Machine  Learning,  Data  
Mining,
 
Cloud
 
Computing,
 
Data-intensive
 
systems
 
for
 
ML,
 
Data
 
Visualization
  
Vellore  Institute  of  Technology,  Vellore  Vellore,  India  Bachelor’s  in  Computer  Science  |  GPA:  9.29/10  |  Merit  Scholarship  Recipient  (Top  10  rank)  Jul.  2018  -  May  2022   
WORK  EXPERIENCE  Wells  Fargo  Charlotte,  NC  Gen  AI  Intern,  Quantitative  Analytics  Program  June  2025  -  Aug.  2025  ●  Built  a  POC  Generative  AI  Question  Answering  system  using  customer  data,  providing  bankers  with  accurate  
context-aware
 
insights
 
and
 
reducing
 
client

outlined in the position.
Beyond my technical expertise, I am a Certified Trainer and a Dale Carnegie Training Excellence
graduate, which has strengthened my communication and interpersonal skills. I’ve delivered technical
sessions at work to 60+ employees, refi